In [ ]:
!git clone https://github.com/dave123981/commerce-recommendation-engine.git
%cd commerce-recommendation-engine
!pip install -r requirements.txt -q

In [ ]:
!pwd

!ls

In [ ]:
!pip install -r requirements.txt

In [ ]:
!pip install -q kagglehub
import kagglehub

!python data/download_data.py          # kagglehub is now the default method
!python data/build_interactions.py

In [ ]:
import pandas as pd

interactions = pd.read_csv("data/interactions.csv", parse_dates=["timestamp"])

print(interactions.shape)
print(interactions.nunique())
print("sparsity:", 1 - len(interactions) / (interactions.user_id.nunique() * interactions.item_id.nunique()))
print("orders per user:", interactions.groupby("user_id").order_id.nunique().describe())
print("purchases per item:", interactions.groupby("item_id").order_id.nunique().describe())

In [ ]:
import data.build_interactions as bi
interactions = bi.build_interactions(min_orders_per_user=10, min_purchases_per_item=100)
interactions.to_csv("data/interactions.csv", index=False)

In [ ]:
from src.metrics import time_based_split

train, test = time_based_split(interactions, test_frac=0.2)
train.to_csv("data/train.csv", index=False)
test.to_csv("data/test.csv", index=False)
print(len(train), len(test))

In [ ]:
import tensorflow as tf
print(tf.config.list_physical_devices('GPU'))

In [ ]:
from src.v5_neural import NeuralRecommender
from src.metrics import evaluate_model

model_v5 = NeuralRecommender(embedding_dim=32, epochs=5).fit(train)
metrics_v5 = evaluate_model(model_v5, test, k=10)
print(metrics_v5)